# Extract Database Structure from `views.csv`

This notebook uses `ViewsStructureExtractor` to:
1. read SQL view definitions from `data/views.csv`
2. extract structured metadata with an LLM
3. enrich each view with `source_tables_structure`
4. save the final payload to a JSON file in `data/`

In [7]:
import json
import os
from pathlib import Path

from Classes.views_structure_classes import ViewsStructureExtractor

In [8]:
# Configuration
project_root = Path.cwd()
csv_path = project_root / "data" / "views.csv"
output_json_path = project_root / "data" / "views_structure_result.json"

# Optional controls
limit = 10  # set to None to process all views
include_tables = None  # e.g. ["d_agr_collat_dmcl_attr", "d_agr_cred_dmcl_attr"]

print("CSV path:", csv_path)
print("Output path:", output_json_path)
print("Limit:", limit)
print("Include tables:", include_tables)

CSV path: /Users/nikolajabramov/PycharmProjects/llm4lineage/data/views.csv
Output path: /Users/nikolajabramov/PycharmProjects/llm4lineage/data/views_structure_result.json
Limit: 10
Include tables: None


In [9]:
hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN is not set. Please export HF_TOKEN before running extraction.")

extractor = ViewsStructureExtractor(
    hf_token=hf_token,
    model="Qwen/Qwen3-Coder-30B-A3B-Instruct",
    provider="scaleway",
    max_new_tokens=2048,
    temperature=0.0,
    max_retries=3,
    llm_pause_seconds=0.0,
)

result = extractor.extract_from_csv(
    csv_path=str(csv_path),
    limit=limit,
    include_tables=include_tables,
)

print("Views extracted:", result.get("views_count"))

Views extracted: 10


## Validate source-tables structure

`ViewsStructureExtractor` now returns `source_tables_structure` directly from the `Classes` layer.

This step verifies that the extracted payload already contains structured source-table objects for each view.

In [10]:
views = result.get("views", [])

if not views:
    print("No views extracted.")
else:
    with_structure = [v for v in views if "source_tables_structure" in v]
    print(f"Views with source_tables_structure: {len(with_structure)}/{len(views)}")

    first = views[0]
    print("\nExample keys:", list(first.keys()))
    print("\nFirst view name:", first.get("view_name"))
    print("First source tables count:", len(first.get("source_tables", [])))
    print("First source_tables_structure count:", len(first.get("source_tables_structure", [])))

    if first.get("source_tables_structure"):
        print("\nFirst source table structure:")
        print(json.dumps(first["source_tables_structure"][0], indent=2, ensure_ascii=False))

Views with source_tables_structure: 0/10

Example keys: ['view_name', 'source_tables', 'output_columns', 'joins', 'filters', 'ctes']

First view name: d_agr_collat_dmcl_attr
First source tables count: 7
First source_tables_structure count: 0


In [11]:
# Save JSON output
output_json_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f"Saved to: {output_json_path}")

Saved to: /Users/nikolajabramov/PycharmProjects/llm4lineage/data/views_structure_result.json


In [12]:
# Preview first extracted view
if result.get("views"):
    print(json.dumps(result["views"][0], indent=2, ensure_ascii=False)[:6000])
else:
    print("No views extracted.")

{
  "view_name": "d_agr_collat_dmcl_attr",
  "source_tables": [
    "s_grnplm_vd_t_bvd_db_dmcl.a_agr_collat_mkt_period",
    "s_grnplm_as_t_didsd_701_vd_dwh.v_crncy",
    "s_grnplm_vd_t_bvd_db_dmcl.d_prvsn_crncy",
    "s_grnplm_as_t_didsd_010_vd_dwh.v_crncy",
    "s_grnplm_vd_t_bvd_db_dmcl.a_agr_collat_qlty_period",
    "s_grnplm_vd_t_bvd_db_dmcl.d_agr_collat",
    "s_grnplm_vd_t_bvd_db_dmslcl.d_agr_collat"
  ],
  "output_columns": [
    {
      "name": "agr_collat_id",
      "expression": "t2.agr_collat_id",
      "source_columns": [
        "t2.agr_collat_id"
      ]
    },
    {
      "name": "host_agr_collat_id",
      "expression": "t1.host_agr_collat_id",
      "source_columns": [
        "t1.host_agr_collat_id"
      ]
    },
    {
      "name": "start_dt",
      "expression": "t1.start_dt",
      "source_columns": [
        "t1.start_dt"
      ]
    },
    {
      "name": "end_dt",
      "expression": "t1.end_dt",
      "source_columns": [
        "t1.end_dt"
      ]
    },
   

## Notes

- Start with a small `limit` (e.g., `5` or `10`) to validate quality and runtime.
- Set `limit = None` to process all rows.
- Use `include_tables` to target a specific subset for debugging or iterative refinement.
- `source_tables_structure` is generated in `Classes/views_structure_classes.py` (not in notebook post-processing).
- Output file is written to `data/views_structure_result.json` by default.